In [1]:
# Pin here for fix of CPU offloading bug; implemented in docker /uv
#%pip install "transformers==4.57.3" "accelerate==1.12.0" "bitsandbytes==0.49.1" "vllm<0.22.0"

In [2]:
# Environment fixes
import os
os.environ["FLASHINFER_DISABLE_VERSION_CHECK"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [3]:
from experiments_pretrained import *
from data import *
from config_record_activations import *
import pickle

/home/dylan/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


In [4]:
# Set seeds
seed = 43
torch.manual_seed(seed);

In [5]:
# Get data: general prompts
dataset = get_data_mmlu(n_samples=n_samples, shuffle_seed=seed)
prompts = format_prompts_mmlu(dataset)

Streaming cais/mmlu (all) (samples: 15000)...


In [6]:
# Get model
model, tokenizer = load_model(model_id, enable_bnb=True)
probe = MoEProbeQwen(model)

Loading checkpoint shards:   0%|          | 0/8 [00:00<?, ?it/s]

MoEHook: Scanning model for routers...
MoEHook: Attached probes to 24 router layers.
MoEHook: Model has 24 routers each with 60 experts and selects k=4 at each layer.
MoEProbe: Qwen1.5-MoE-A2.7B model also has shared expert with intermediate size: 5632 (Equivalent to ~4 routed experts)


In [ ]:
# Record activations over generalized MMLU questions
results = get_activations_mmlu(model, tokenizer, dataset, probe=probe, max_new_tokens=max_new_tokens, batch_size=batch_size)
#results = get_activations_mmlu(model, tokenizer, dataset, probe=probe, max_new_tokens=max_new_tokens, batch_size=1)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Generating responses 1-8/14042...


/home/dylan/.local/lib/python3.10/site-packages/bitsandbytes/backends/cuda/ops.py:464: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Batch inference took 51.251s (6.406s/sample).
Generating responses 9-16/14042...
Batch inference took 50.590s (6.324s/sample).
Generating responses 17-24/14042...
Batch inference took 50.498s (6.312s/sample).
Generating responses 25-32/14042...
Batch inference took 49.918s (6.240s/sample).
Generating responses 33-40/14042...
Batch inference took 51.086s (6.386s/sample).
Generating responses 41-48/14042...
Batch inference took 51.804s (6.476s/sample).
Generating responses 49-56/14042...
Batch inference took 51.978s (6.497s/sample).
Generating responses 57-64/14042...
Batch inference took 52.256s (6.532s/sample).
Generating responses 65-72/14042...
Batch inference took 51.393s (6.424s/sample).
Generating responses 73-80/14042...
Batch inference took 52.441s (6.555s/sample).
Generating responses 81-88/14042...
Batch inference took 52.510s (6.564s/sample).
Generating responses 89-96/14042...
Batch inference took 52.788s (6.599s/sample).
Generating responses 97-104/14042...
Batch inference 

In [ ]:
# Save results
with open(results_file, 'wb') as file:
    pickle.dump(results, file)

In [ ]:
with open("results_nonbatched.pkl", 'rb') as file:
    results_nonbatched = pickle.load(file)
with open("results_batched.pkl", 'rb') as file:
    results_batched = pickle.load(file)

prompt_id = 1
print(results_nonbatched[prompt_id]['prompt'])
print(results_nonbatched[prompt_id]['subject'])
print(results_nonbatched[prompt_id]['probs'].shape)
print(results_nonbatched[prompt_id]['active_experts'][0, :, 1])
print(results_batched[prompt_id]['prompt'])
print(results_batched[prompt_id]['subject'])
print(results_batched[prompt_id]['probs'].shape)
print(results_batched[prompt_id]['active_experts'][0, :, 1])